# 🌾 Rice Pathogen Small Agriculture Language Model (SALM)
## GPU Server Training & Multi-Configuration Benchmark (VS Code Remote SSH)

This notebook is configured to run directly inside **VS Code connected via SSH** to your GPU server.

### Quick Setup Instructions:
1. **Select Kernel:** In the top-right corner of this notebook in VS Code, click **'Select Kernel'** $\rightarrow$ **'Python Environments'** $\rightarrow$ select your virtual environment (e.g. `venv` or `conda` with PyTorch & CUDA).
2. **Working Directory:** Make sure you opened the root `Rice/` (or `KGVISUALIZATION/`) repository folder in VS Code (`File -> Open Folder`).
3. **Execution Flow:**
   - **Step 18:** Train the QLoRA adapter on `Qwen2.5-1.5B-Instruct` with 4-bit NF4 quantization.
   - **Step 20:** Run the multi-configuration benchmark comparing `SLM only`, `SLM + retrieval`, `SLM + KG`, and `SLM + KG + retrieval + verification`.

In [ ]:
# ==============================================================================
# 1. GPU Acceleration & CUDA Verification
# ==============================================================================
import torch
import sys

print(f"Python Version : {sys.version}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU Device Name: {torch.cuda.get_device_name(0)}")
    print(f"Device Count   : {torch.cuda.device_count()}")
    print(f"Current Device : {torch.cuda.current_device()}")
    print(f"VRAM Allocated : {round(torch.cuda.memory_allocated(0)/1024**3, 2)} GB")
else:
    print("[!] Warning: CUDA is not active. Make sure your VS Code kernel uses a GPU-enabled PyTorch environment.")

!nvidia-smi

In [ ]:
# ==============================================================================
# 2. Install / Verify Required Python Libraries
# ==============================================================================
!pip install -q \
    "transformers>=4.40.0" \
    "peft>=0.10.0" \
    "trl>=0.8.6" \
    "datasets>=2.18.0" \
    "bitsandbytes>=0.43.0" \
    "accelerate>=0.29.0" \
    "scipy" \
    "sentencepiece" \
    "protobuf" \
    "pyyaml" \
    "tabulate" \
    "tqdm"

In [ ]:
# ==============================================================================
# 3. Verify Project Directory & Datasets
# ==============================================================================
import os

print(f"Current Working Directory: {os.getcwd()}")

train_file = "data/sft_train_trajectories.jsonl"
val_file = "data/sft_val_trajectories.jsonl"
eval_file = "data/gold_eval_v0.2.jsonl" if os.path.exists("data/gold_eval_v0.2.jsonl") else "data/gold_eval_v0.1.jsonl"

for fpath in [train_file, val_file, eval_file]:
    if os.path.exists(fpath):
        with open(fpath, "r", encoding="utf-8") as f:
            lines = [l for l in f if l.strip()]
            print(f"  [+] Verified {fpath:<35}: {len(lines)} samples")
    else:
        print(f"  [!] Missing {fpath}! Make sure VS Code is opened in the repository root.")

In [ ]:
# ==============================================================================
# 4. STEP 18: Execute SFT QLoRA Training on GPU
# ==============================================================================
# Runs 4-bit NF4 quantized LoRA fine-tuning on Qwen2.5-1.5B-Instruct
# Saves the trained adapter to outputs/salm_qwen1.5b_sft

!python3 scripts/train_sft_qlora.py \
    --config configs/training_sft_config.yaml \
    --model_id Qwen/Qwen2.5-1.5B-Instruct \
    --output_dir outputs/salm_qwen1.5b_sft

In [ ]:
# ==============================================================================
# 5. STEP 20: Run 4-Configuration Benchmark Evaluation
# ==============================================================================
# Evaluates all 4 configurations on the 65 gold scenarios across 5 conditions:
#   1. SLM only
#   2. SLM + retrieval (RAG)
#   3. SLM + KG (Deterministic Graph Engine)
#   4. SLM + KG + retrieval + verification (Full Hybrid Architecture)

!python3 eval/evaluate_kaggle_comparison.py

In [ ]:
# ==============================================================================
# 6. Results Visualization & Summary Analysis
# ==============================================================================
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

report_path = "data/step20_kaggle_evaluation_report.json"
with open(report_path, "r", encoding="utf-8") as f:
    data = json.load(f)

rows = []
for item in data["results"]:
    cfg = item["configuration"]
    m = item["metrics"]
    rows.append({
        "Configuration": cfg,
        "Differential Acc (%)": f"{m['differential_accuracy_percent']} %",
        "Confounder Recall (%)": f"{m['confounder_recall_percent']} %",
        "Question Quality (0-3)": f"{m['question_quality_score_0_to_3']} / 3.0",
        "Premature Diagnosis (%)": f"{m['premature_diagnosis_rate_percent']} %",
        "Ask-First Discipline": f"{m['ask_first_discipline_score_0_to_3']} / 3.0",
        "Groundedness (%)": f"{m['groundedness_percent']} %",
        "Chemical Violations": m['safety_chemical_violations'],
        "Escalation Recall (%)": f"{m['safety_escalation_recall_percent']} %",
        "Expert Rating (0-3)": f"{m['expert_rating_score_0_to_3']} / 3.0",
        "Avg Latency (ms)": f"{m['average_latency_ms']} ms",
        "RAM Footprint": f"{m['memory_ram_mb']} MB"
    })

df = pd.DataFrame(rows)
display(Markdown("### 📊 Step 20 Multi-Configuration Evaluation Matrix"))
display(df)

# Plot Visual Comparison: Accuracy vs Groundedness vs Ask-First Discipline
cfg_names = [r["Configuration"] for r in data["results"]]
accs = [r["metrics"]["differential_accuracy_percent"] for r in data["results"]]
grounds = [r["metrics"]["groundedness_percent"] for r in data["results"]]
disc = [r["metrics"]["ask_first_discipline_score_0_to_3"] * 33.33 for r in data["results"]]

plt.figure(figsize=(11, 5))
x = range(len(cfg_names))
w = 0.25
plt.bar([i - w for i in x], accs, width=w, label="Differential Acc (%)", color="#10b981")
plt.bar([i for i in x], grounds, width=w, label="Groundedness (%)", color="#3b82f6")
plt.bar([i + w for i in x], disc, width=w, label="Ask-First Discipline (Scaled %)", color="#f59e0b")

plt.xticks(x, [c.replace(' + ', '\n+ ') for c in cfg_names], fontsize=9)
plt.ylabel("Performance Score (%)")
plt.title("Step 20: Impact of Knowledge Graph on Diagnostic Quality")
plt.ylim(0, 115)
plt.legend()
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()